# Lab 05: Column Generation

**Author:** Bo Tang

Solve a large linear program by generating useful variables as they are needed.

## Learning Objectives

- Formulate a restricted master problem and a pricing problem.
- Use reduced costs to decide when to add a column.
- Compare an LP relaxation with an integer solution over generated patterns.

## Setup

Use a Python kernel with `gurobipy` and `numpy` installed and a working Gurobi license. See the [README](README.md#getting-started) for setup instructions.

Run the optional installation cell only if packages are missing. Then run the imports and work through the notebook in order. Cells labeled `TODO` are student exercises; complete them before running dependent cells. Worked examples retain their sample outputs.

In [ ]:
# Optional: run only if these packages are missing from this kernel.
%pip install gurobipy numpy

In [1]:
import gurobipy as gp
import numpy as np
from gurobipy import GRB

## Column Generation Review

Column Generation is often used to solve large-scale linear programming problems where the number of decision variables is extremely large, making it intractable to enumerate and include all variables in the model initially.

This method is particularly useful in problems like cutting stock, vehicle routing, and crew scheduling, where only a small subset of the decision variables (columns) are expected to be active in the optimal solution.

The key idea behind Column Generation is to start with a smaller, manageable subset of decision variables (columns) in the **Master Problem** and iteratively add new, more promising variables by solving a **Subproblem**. The Subproblem identifies which variables (columns) have the potential to improve the solution of the Master Problem by evaluating their **reduced cost**.

Consider a large-scale linear program in the following form, where we want to maximize the objective function:

$$
\max_x \mathbf{c}^\top \mathbf{x}
$$
Subject to
$$
\mathbf{A} \mathbf{x} \leq \mathbf{b}
$$
$$
\mathbf{x} \geq \mathbf{0}
$$

Here, $\mathbf{x}$represents the decision variables,$\mathbf{c}$the cost coefficients, and$\mathbf{A} \mathbf{x} \leq \mathbf{b}$ represents the constraints.

### Master Problem

In Column Generation, we start by solving a restricted version of this problem, known as the **Restricted Master Problem (RMP)**, which only includes a subset of decision variables $\{ x_1, x_2, \cdots, x_k \}$and the corresponding constraint columns$\{ \mathbf{a}_1, \mathbf{a}_2, \cdots, \mathbf{a}_k \}$.

Thus, the Master is just the same as the large model formulation, but using the variables found in the previous $k$ iterations.

$$
\max_x \sum_{j=1}^k c_j x_j
$$
Subject to
$$
\sum_{j=1}^k \mathbf{a}_j x_j \leq \mathbf{b}
$$
$$
x_j \geq 0 \quad \forall j
$$

### Pricing Problem

Let $\lambda$be the restricted master's dual solution. The reduced cost of a candidate column$a_j$ is

$$
\bar c_j = c_j - \lambda^\top a_j.
$$

For the **maximization** formulation above, search for a column with positive reduced cost. Equivalently, minimize $\lambda^\top a_j-c_j$ and add the column if this pricing objective is negative. The reduced cost and its negative must not be confused.

For a **minimization** master, search for negative reduced cost instead. Once the restricted master is optimal and an exact pricing solve proves that no improving column exists, its solution is optimal for the full LP. Use a numerical tolerance in practice.

See the [MIT column-generation notes](https://ocw.mit.edu/courses/1-224j-carrier-systems-fall-2003/132da550d77666a74f5417e3f7dc4d1a_lecture10.pdf) for another worked formulation.

**Discussion questions:**

- What happens in a minimization master?

### Integer Programming

Unfortunately, CG does not solve ILP problems, so we proceed as follows:

1. Relax the integer variables and solve the relaxed problem using Column Generation as a linear program (LP).
2. Reintroduce the integer variables to the master problem to obtain a solution.

It is important to note that after relaxing the integer constraints and reintroducing them, Column Generation CANNOT guarantee global optimality for integer linear problems.

#### Optional: Branch-and-Price

**Branch-and-price** is an algorithm that combines Column Generation with the **Branch-and-Bound** method, specifically designed to handle complex problems with integer variables. In this approach, Column Generation is embedded within the Branch-and-Bound framework to solve the linear relaxation of each node.

In branch-and-price, the algorithm first uses Column Generation to solve the relaxed linear problem, then introduces branching operations at each node of the search tree to handle the integer variables. Unlike traditional Branch-and-Bound, Branch & Price only generates columns relevant to the current branch, which makes it more efficient for large-scale problems.

## Example 1: Cutting Stock

You sell copper wire to the retail market in lengths of 3', 5', and 9'. The respective demand for these lengths is 25, 20, and 15 units. You buy copper wire on the wholesale market in 107' lengths. The objective is to determine how many 107' lengths to buy and how to cut them to satisfy the demand with minimal cost.

In [2]:
# Available lengths
lengths = [3, 5, 9]
# Demand
demand = [25, 20, 15]

### Exact Large Model Formulation

We can model this problem as an integer linear programming (ILP) problem.

Let $x_j$ be the number of times cutting pattern$j$ is used, where each pattern represents a possible way of cutting the 107' wholesale length into lengths of 3', 5', and 9'.

The optimization model aims to minimize the total number of wholesale copper wire lengths used:

$$
\min_x \sum_{j=1}^k x_j
$$
Subject to
$$
\sum_{j=1}^k a_{3j} x_j \geq 25
$$
$$
\sum_{j=1}^k a_{5j} x_j \geq 20
$$
$$
\sum_{j=1}^k a_{9j} x_j \geq 15
$$
$$
x_j \in \mathbb{Z}_{\geq 0} \quad \forall j
$$

**Discussion questions:**

- How many cutting patterns do we have?

In [3]:
import numpy as np

# Wholesale length of copper wire
wholesale_length = 107

# Generate all combinations of cutting patterns
def generate_combinations():
    combinations = []
    for a3 in range(wholesale_length // lengths[0] + 1):
        for a5 in range(wholesale_length // lengths[1] + 1):
            for a9 in range(wholesale_length // lengths[2] + 1):
                if (a3 * lengths[0] + a5 * lengths[1] + a9 * lengths[2]) <= wholesale_length:
                    combinations.append((a3, a5, a9))
    return combinations

# Generate all possible cutting patterns
patterns = generate_combinations()

# Display the number of patterns and a sample
print("Total number of valid cutting patterns:", len(patterns))
print("Sample patterns:", patterns[:10])  # Show first 10 patterns

Total number of valid cutting patterns: 1921
Sample patterns: [(0, 0, 0), (0, 0, 1), (0, 0, 2), (0, 0, 3), (0, 0, 4), (0, 0, 5), (0, 0, 6), (0, 0, 7), (0, 0, 8), (0, 0, 9)]


We can solve this ILP problem.

In [4]:
# Create Gurobi model
m = gp.Model("cutting_stock")

# Decision variables, one for each cutting pattern
x = m.addVars(len(patterns), vtype=GRB.INTEGER, name="x")

# Objective: Minimize the total number of wires used
m.setObjective(gp.quicksum(x[i] for i in range(len(patterns))), GRB.MINIMIZE)

# Demand constraints for each length 3 5 9
m.addConstr(gp.quicksum(patterns[i][0] * x[i] for i in range(len(patterns))) >= demand[0], "Demand_3ft")
m.addConstr(gp.quicksum(patterns[i][1] * x[i] for i in range(len(patterns))) >= demand[1], "Demand_5ft")
m.addConstr(gp.quicksum(patterns[i][2] * x[i] for i in range(len(patterns))) >= demand[2], "Demand_9ft")

# Optimize the model
m.optimize()

# Display the results
if m.status == GRB.OPTIMAL:
    print("\nOptimal solution found:")
    print(f"Minimum number of wires needed: {m.objVal}\n")
    print("Cutting patterns used:")
    for i in range(len(patterns)):
        if x[i].x > 0:
            print(f"Pattern {patterns[i]}: used {x[i].x} times")
else:
    print("No optimal solution found.")

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (win64 - Windows 11+.0 (26300.2))


Optimize a model with 3 rows, 1921 columns and 4970 nonzeros (Min)
Model fingerprint: 0x21c94254
Model has 1921 linear objective coefficients
Variable types: 0 continuous, 1921 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 4e+01]
  Objective range  [1e+00, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+01, 3e+01]

Found heuristic solution: objective 35.0000000
Presolve removed 0 rows and 68 columns
Presolve time: 0.00s
Presolved: 3 rows, 1853 columns, 4830 nonzeros
Found heuristic solution: objective 3.0000000
Variable types: 0 continuous, 1853 integer (2 binary)

Root relaxation: cutoff, 4 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

     0     0     cutoff    0         3.00000    3.00000  0.00%     -    0s

Explo

## Column Generation for Cutting Stock

The restricted master chooses how often to use each available pattern. The pricing problem generates a new pattern using the master LP's dual prices. Keep the master variables continuous while generating columns.

### Initial Pattern

We will start with a simple initial cutting pattern:
$$
\begin{pmatrix}
1 \\
1 \\
11
\end{pmatrix}
$$

In [5]:
init_pattern = [1, 1, 11]

### Exercise: Restricted Master Problem

Complete the decision variables and demand constraints. Use continuous nonnegative variables so that the demand constraints have LP dual values. Keep their names consistent with the `getConstrByName` calls.

In [ ]:
def solve_master_problem(patterns):
    # Create Gurobi model
    m = gp.Model("master")
    # Turn off log
    m.Params.outputFlag = 0
    # TODO: decision variables for the number of times each pattern is used
    x = m.addVars
    # Objective: Minimize the number of 107' copper wires used
    m.setObjective(gp.quicksum(x[i] for i in range(len(patterns))), GRB.MINIMIZE)
    # TODO: constraints to meet demand for each length (3', 5', 9')
    m.addConstr
    # Optimize the master
    m.optimize()
    # Objective value
    objval = m.objVal
    # Return the dual variables (shadow prices) from the demand constraints
    λ = [m.getConstrByName("Demand_3ft").Pi,
         m.getConstrByName("Demand_5ft").Pi,
         m.getConstrByName("Demand_9ft").Pi]
    return objval, λ

# Solve master problem
objval, λ = solve_master_problem([init_pattern])

### Exercise: Pricing Problem

For a pattern $a=(a_3,a_5,a_9)$, the reduced cost is $1-\lambda^\top a$. Complete the integer knapsack model:

$$
\max\ \lambda^\top a-1
\quad\text{subject to}\quad
3a_3+5a_5+9a_9\leq107,\qquad a\in\mathbb{Z}_{\geq0}^3.
$$

Return the **improvement score** $\lambda^\top a-1$ and the new pattern. The loop below adds a pattern when this score is positive; the variable named `rcost` in that scaffold stores the negative of the reduced cost.

In [ ]:
def solve_subproblem(λ):
    # Create the subproblem to find new cutting patterns
    m = gp.Model("Subproblem")
    # Turn off log
    m.Params.outputFlag = 0
    # TODO: decision variables for the number of 3', 5', and 9' lengths

    # TODO: objective: maximize the improvement score (dual-weighted pattern value minus 1)

    # TODO: constraint: less than or equal to whole length

    # Optimize the subproblem
    m.optimize()
    # Objective value
    objval = m.objVal
    # TODO: get new pattern
    new_pattern =
    return objval, new_pattern

objval, new_pattern = solve_subproblem(λ)
print("\nObjective value:", objval)
print("New pattern:", new_pattern)

### Column Generation Iteration

We will now implement the Column Generation algorithm to iteratively solve the master and Subproblem. The algorithm will add new patterns until no further improvements can be made.

In [ ]:
# Exercise: complete the preceding TODOs before running this cell.
def column_generation(init_pattern):
    # Init patterns
    patterns = [init_pattern]
    cnt = 0
    while True:
        # Solve master
        objval, λ = solve_master_problem(patterns)
        # Solve subproblem
        rcost, new_pattern = solve_subproblem(λ)
        # If no improvement, break
        if rcost <= 0:
            break
        # Add new pattern
        patterns.append(new_pattern)
        cnt += 1
        print(f"Iteration {cnt}: New pattern {new_pattern} with objective value {objval}")
    return patterns

patterns = column_generation(init_pattern)
# Display the number of patterns and a sample
print("Total number of valid cutting patterns:", len(patterns))
print("Patterns:", patterns)

### Integer Solution over Generated Patterns

Reintroduce integrality and solve the master over the generated patterns. This produces a feasible integer solution, but it does not generally certify the optimum over all possible patterns. Compare it with the enumerated baseline.

In [ ]:
# Exercise: complete the preceding TODOs before running this cell.
# Create Gurobi model
m = gp.Model("cutting_stock")

# Decision variables, one for each cutting pattern
x = m.addVars(len(patterns), vtype=GRB.INTEGER, name="x")

# Objective: Minimize the total number of wires used
m.setObjective(gp.quicksum(x[i] for i in range(len(patterns))), GRB.MINIMIZE)

# Demand constraints for each length 3 5 9
m.addConstr(gp.quicksum(patterns[i][0] * x[i] for i in range(len(patterns))) >= demand[0], "Demand_3ft")
m.addConstr(gp.quicksum(patterns[i][1] * x[i] for i in range(len(patterns))) >= demand[1], "Demand_5ft")
m.addConstr(gp.quicksum(patterns[i][2] * x[i] for i in range(len(patterns))) >= demand[2], "Demand_9ft")

# Optimize the model
m.optimize()

# Display the results
print("\nOptimal solution found:")
print(f"Minimum number of wires needed: {m.objVal}\n")
print("Cutting patterns used:")
for i in range(len(patterns)):
    if x[i].x > 0:
        print(f"Pattern {patterns[i]}: used {x[i].x} times")

We can observe that the solution obtained through Column Generation may not always be the optimal solution for integer linear problems.

## Example 2: Clothing Company Shipment

A clothing company is in the process of creating a set of new shipments $s \in S$. Each shipment combines a set of available clothing items from the set $F$. Each clothing item $i \in F$has an expected sales revenue of$r_i$dollars, handling cost of$c_i$, and the total number of units available of item $i$is$d_i$. Also, the maximum number of distinct shipments that can be made is equal to $K$. Finally, for each shipment, the total handling cost should not exceed 50\% of the expected revenue.
You have been hired by this company to help optimize the shipments that should be sent out in order to maximize the total expected revenue.

The company has decided to model the problem in the following way:
Let $S$ be the set of all feasible shipment patterns that can be created.
Let $a_{is}$be the number of clothing item$i$in shipment$s$.
Let $r_s = \sum_{i \in F} a_{is} r_i$be the expected revenue from shipment$s$.

Let variables $x_s$be equal to 1 if shipment$s$ will be sent, and 0 otherwise.

Then the problem formulation follows:
$$
\max \sum_{s \in S} r_s x_s
$$
Subject to:
$$
\sum_{s \in S} a_{is} x_s \leq d_i \quad \forall i \in F
$$
$$
\sum_{s \in S} x_s \leq K
$$
$$
x_s \in \{0, 1\} \quad \forall s \in S
$$

The first set of constraints ensures that no more than the available amount is sent out for each item $i$.
The last constraint ensures that no more than $K$ shipments are sent.

Suppose you are given dual variables $\alpha_i$for the first set of constraints, and$\beta$ for the last constraint.

Now consider solving the LP relaxation of this problem and applying column generation.

- Write down a formula for the reduced cost for a variable $x_s$.
- Assuming that you are using column generation for solving the problem, explain how you might set up your pricing problem.
- Explain what the decision variables will be and write down the objective function and constraints.

**Your formulation:** Write the reduced cost, pricing variables, objective, and constraints here.

## Summary

- Use continuous master variables to obtain dual prices during column generation.
- Match the pricing objective and stopping rule to the master objective direction.
- An integer solve over generated columns is not generally a global integer optimality certificate.